In [0]:
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()


def _notebook_dir():
    """Workspace folder of the notebook this code is running in."""
    path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    return "/Workspace" + os.path.dirname(path)


def create_job_config(job_name, job_tasks, job_parameters=None, base_path=None, replace=False):
    """Create a Lakeflow job from a list of task dicts.

    job_tasks:      [{'task_name': ..., 'file_path': ..., 'depends_on': None | [{'task_key': ...}]}]
    job_parameters: [{'name': ..., 'default': ...}]
    base_path:      folder that each file_path is relative to (default: this notebook's folder)
    replace:        if True, delete any existing job with the same name first
    """
    existing = list(w.jobs.list(name=job_name))
    if existing:
        if not replace:
            raise ValueError(f"Job '{job_name}' already exists (job_id={existing[0].job_id}).")
        for j in existing:
            w.jobs.delete(job_id=j.job_id)
            print(f"Deleted existing job {j.job_id}")
    print(f"Job name is unique. Creating the job {job_name}...")

    base_path = (base_path or _notebook_dir()).rstrip("/")
    print(f"Using the following path to reference the Files: {base_path}/")

    tasks = [
        jobs.Task(
            task_key=t["task_name"],
            notebook_task=jobs.NotebookTask(
                notebook_path=f"{base_path}/{t['file_path'].lstrip('/')}"
            ),
            depends_on=[
                jobs.TaskDependency(task_key=d["task_key"])
                for d in (t.get("depends_on") or [])
            ] or None,
        )
        for t in job_tasks
    ]

    job = w.jobs.create(
        name=job_name,
        tasks=tasks,
        parameters=[
            jobs.JobParameterDefinition(name=p["name"], default=p["default"])
            for p in (job_parameters or [])
        ],
    )
    print(f"Created job_id={job.job_id}")
    return job.job_id